# UFC Fight Data Explorer
Explore the raw dataframe with round-by-round data

In [ ]:
import polars as pl
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set display options
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Load the parquet file
df = pl.read_parquet('fight_snapshots_all_with_rounds.parquet')
print(f"Loaded {len(df)} fights")
print(f"Columns: {len(df.columns)}")

## 1. Basic Overview

In [ ]:
# View column names
print("Column names:")
for col in df.columns:
    print(f"  - {col}")

In [ ]:
# View first few rows (basic columns only)
basic_cols = ['fight_id', 'fight_date', 'fighter1_id', 'fighter2_id', 'winner_id', 'weight_class', 'prior_cnt_f1', 'prior_cnt_f2']
df.select(basic_cols).head(10)

In [ ]:
# Distribution of prior fight counts
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Fighter 1 distribution
df['prior_cnt_f1'].to_pandas().hist(bins=50, ax=ax1, color='skyblue', edgecolor='black')
ax1.set_xlabel('Number of Prior Fights')
ax1.set_ylabel('Frequency')
ax1.set_title('Fighter 1: Prior Fight Count Distribution')
ax1.axvline(df['prior_cnt_f1'].median(), color='red', linestyle='--', label=f'Median: {df["prior_cnt_f1"].median()}')
ax1.legend()

# Fighter 2 distribution
df['prior_cnt_f2'].to_pandas().hist(bins=50, ax=ax2, color='lightcoral', edgecolor='black')
ax2.set_xlabel('Number of Prior Fights')
ax2.set_ylabel('Frequency')
ax2.set_title('Fighter 2: Prior Fight Count Distribution')
ax2.axvline(df['prior_cnt_f2'].median(), color='red', linestyle='--', label=f'Median: {df["prior_cnt_f2"].median()}')
ax2.legend()

plt.tight_layout()
plt.show()

## 2. Explore a Single Fight's History

In [ ]:
# Pick a random fight with decent history
sample_fight = df.filter((pl.col('prior_cnt_f1') >= 10) & (pl.col('prior_cnt_f2') >= 10)).sample(1)

print("Sample Fight:")
print(f"  Fight ID: {sample_fight['fight_id'][0]}")
print(f"  Date: {sample_fight['fight_date'][0]}")
print(f"  Weight Class: {sample_fight['weight_class'][0]}")
print(f"  Fighter 1 Prior Fights: {sample_fight['prior_cnt_f1'][0]}")
print(f"  Fighter 2 Prior Fights: {sample_fight['prior_cnt_f2'][0]}")

In [ ]:
# Extract fighter 1's fight history
f1_history = sample_fight['prior_f1'][0]

print(f"\nFighter 1 has {len(f1_history)} prior fights")
print("\nMost recent 3 fights:")
for i, fight in enumerate(f1_history[-3:]):
    print(f"\n  Fight {i+1}:")
    print(f"    Date: {fight['fight_date']}")
    print(f"    Result: {fight['result']}")
    print(f"    Method: {fight.get('method', 'N/A')}")
    print(f"    Opponent: {fight['opponent_id']}")
    if 'sig_str_landed' in fight:
        print(f"    Sig Strikes: {fight['sig_str_landed']}/{fight['sig_str_attempts']}")
    if 'rounds' in fight and fight['rounds'] is not None:
        print(f"    Rounds: {len(fight['rounds'])} rounds")

## 3. Explore Round-by-Round Data

In [ ]:
# Get a fight with round data
f1_history = sample_fight['prior_f1'][0]

# Find a fight with rounds
fight_with_rounds = None
for fight in f1_history:
    if 'rounds' in fight and fight['rounds'] is not None and len(fight['rounds']) > 0:
        fight_with_rounds = fight
        break

if fight_with_rounds:
    print("Fight with round data found!")
    print(f"  Date: {fight_with_rounds['fight_date']}")
    print(f"  Result: {fight_with_rounds['result']}")
    print(f"  Total Rounds: {len(fight_with_rounds['rounds'])}")
    print("\n  Round-by-round breakdown:")
    for round_data in fight_with_rounds['rounds']:
        print(f"\n    Round {round_data['round_number']}:")
        print(f"      Sig Strikes: {round_data['sig_str_landed']}/{round_data['sig_str_attempts']}")
        print(f"      Total Strikes: {round_data['total_str_landed']}/{round_data['total_str_attempts']}")
        print(f"      Takedowns: {round_data['td_landed']}/{round_data['td_attempts']}")
        print(f"      Knockdowns: {round_data['kd']}")
        print(f"      Control Time: {round_data['ctrl_time_s']}s")
else:
    print("No fights with round data found in this sample")

## 4. Visualize Round Performance

In [ ]:
# Plot strike accuracy across rounds
if fight_with_rounds:
    rounds = []
    accuracy = []
    sig_strikes = []
    
    for round_data in fight_with_rounds['rounds']:
        rounds.append(round_data['round_number'])
        if round_data['sig_str_attempts'] and round_data['sig_str_attempts'] > 0:
            acc = (round_data['sig_str_landed'] / round_data['sig_str_attempts']) * 100
            accuracy.append(acc)
            sig_strikes.append(round_data['sig_str_landed'])
        else:
            accuracy.append(0)
            sig_strikes.append(0)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Strike accuracy
    ax1.plot(rounds, accuracy, marker='o', linewidth=2, markersize=8, color='steelblue')
    ax1.set_xlabel('Round')
    ax1.set_ylabel('Strike Accuracy (%)')
    ax1.set_title('Striking Accuracy Across Rounds')
    ax1.grid(True, alpha=0.3)
    ax1.set_xticks(rounds)
    
    # Significant strikes landed
    ax2.bar(rounds, sig_strikes, color='coral', edgecolor='black')
    ax2.set_xlabel('Round')
    ax2.set_ylabel('Sig Strikes Landed')
    ax2.set_title('Significant Strikes Landed per Round')
    ax2.set_xticks(rounds)
    
    plt.tight_layout()
    plt.show()

## 5. Fighter Stats Comparison

In [ ]:
# Compare fighter attributes
comparison_cols = ['f1_height_in', 'f2_height_in', 'f1_reach_in', 'f2_reach_in', 'f1_win', 'f2_win', 'f1_loss', 'f2_loss']
sample_fight.select(comparison_cols)

## 6. Data Quality Check

In [ ]:
# Check for missing data
print("Missing data by column:")
for col in df.columns:
    null_count = df[col].null_count()
    if null_count > 0:
        pct = (null_count / len(df)) * 100
        print(f"  {col}: {null_count} ({pct:.1f}%)")

In [ ]:
# Check fights with round data
def has_round_data(prior_fights):
    if prior_fights is None or len(prior_fights) == 0:
        return False
    for fight in prior_fights:
        if 'rounds' in fight and fight['rounds'] is not None and len(fight['rounds']) > 0:
            return True
    return False

f1_with_rounds = sum(1 for pf in df['prior_f1'] if has_round_data(pf))
f2_with_rounds = sum(1 for pf in df['prior_f2'] if has_round_data(pf))

print(f"\nFights where Fighter 1 has round data: {f1_with_rounds} / {len(df)} ({f1_with_rounds/len(df)*100:.1f}%)")
print(f"Fights where Fighter 2 has round data: {f2_with_rounds} / {len(df)} ({f2_with_rounds/len(df)*100:.1f}%)")

## 7. Export Sample for Excel
(For very quick viewing in Excel)

In [ ]:
# Export basic columns to CSV for Excel viewing
basic_export = df.select([
    'fight_id', 'fight_date', 'fighter1_id', 'fighter2_id', 'winner_id',
    'weight_class', 'prior_cnt_f1', 'prior_cnt_f2',
    'f1_height_in', 'f2_height_in', 'f1_reach_in', 'f2_reach_in',
    'f1_win', 'f1_loss', 'f2_win', 'f2_loss'
]).head(100)

basic_export.write_csv('sample_fights_basic.csv')
print("Exported 100 sample fights to sample_fights_basic.csv")